# Kaggle Deep Learning Baselines

This notebook trains and evaluates two deep learning models, which are TextCNN and BiLSTM, on the processed Kaggle human and LLM-generated phishing email dataset. Both models use pretrained GloVe 100-dimensional embeddings (frozen) and are evaluated using the same metrics schema as all other models in this project for direct comparison.

## 1. Import Libraries

All libraries needed for loading data, building and training deep learning models, and evaluating and saving results are imported here.

In [ ]:
import os
import time
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence
from collections import Counter

# Metrics for evaluating model performance
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

## 2. Reproducibility and Configuration

A fixed random seed is set across all libraries to ensure reproducible results. All key hyperparameters and file paths are defined here in one place.

In [ ]:
SEED = 42

def set_seed(seed):
    # Fix all sources of randomness for reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# File paths
data_dir    = Path("../data/processed/kaggle/")
glove_path  = Path("../data/glove/glove.6B.100d.txt")
results_dir = Path("../results/")
results_dir.mkdir(parents=True, exist_ok=True)

# Hyperparameters — consistent with meajor_dl_baselines.ipynb where applicable
EMBED_DIM        = 100
MAX_LEN          = 200
VOCAB_SIZE       = 20000
BATCH_SIZE       = 32
LEARNING_RATE    = 1e-3
NUM_EPOCHS       = 10
DROPOUT          = 0.5

# TextCNN: parallel filters capture bigram, trigram, and 4-gram patterns
CNN_FILTER_SIZES = [2, 3, 4]
CNN_NUM_FILTERS  = 128

# BiLSTM
LSTM_HIDDEN      = 128
LSTM_LAYERS      = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 3. Load Processed Data

The train and test Parquet files produced by `kaggle_preparation.ipynb` are loaded here. These contain the cleaned `text` and `label` columns only.

In [ ]:
# Load the 60% training split
train_df = pd.read_parquet(data_dir / "kaggle_train_60.parquet")

# Load the 40% test split
test_df  = pd.read_parquet(data_dir / "kaggle_test_40.parquet")

print(f"Train size:  {len(train_df)} rows")
print(f"Test size:   {len(test_df)} rows")
print(f"Columns:     {train_df.columns.tolist()}")

## 4. Verify Loaded Data

A quick check of label distributions and a sample of the text field to confirm the data loaded correctly before building the vocabulary.

In [ ]:
print("Train label distribution:")
print(train_df["label"].value_counts())

print("\nTest label distribution:")
print(test_df["label"].value_counts())

# Preview a few rows to confirm the text field looks correct
train_df[["text", "label"]].head(3)

## 5. Tokenisation and Vocabulary

The text is split on whitespace, it is already lowercased and cleaned from the preparation notebook. A vocabulary of the most frequent tokens is built from the training set only to avoid data leakage.

In [ ]:
def simple_tokenise(text):
    # Text is already lowercased; split on whitespace only
    return str(text).split()

# Build vocabulary from training data only — no test set leakage
all_tokens = []
for text in train_df["text"]:
    all_tokens.extend(simple_tokenise(text))

token_counts = Counter(all_tokens)

# Index 0 = <PAD> (padding), Index 1 = <UNK> (unknown tokens)
most_common = [tok for tok, _ in token_counts.most_common(VOCAB_SIZE - 2)]
vocab = {"<PAD>": 0, "<UNK>": 1}
for tok in most_common:
    vocab[tok] = len(vocab)

print(f"Vocabulary size: {len(vocab)}")


def encode(text, max_len):
    # Convert text to a fixed-length integer sequence
    tokens = simple_tokenise(text)[:max_len]
    ids    = [vocab.get(t, vocab["<UNK>"]) for t in tokens]
    # Pad with zeros up to max_len if the sequence is shorter
    ids   += [0] * (max_len - len(ids))
    # Also return actual length for BiLSTM packed sequence
    return ids, min(len(tokens), max_len)

## 6. Load GloVe Embeddings

Pretrained GloVe 100-dimensional vectors are loaded and aligned to the vocabulary built in Section 5. Tokens not found in GloVe are initialised with small random values. The embedding matrix is frozen during training to reduce overfitting on the relatively small dataset.